In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import zipfile
from google.colab import drive
from sklearn.cluster import MiniBatchKMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# -----------------------------
# Helper: Mount & extract dataset from Drive
# -----------------------------
def prepare_dataset_from_drive(drive_path, local_extract_path, dataset_name):
    try:
        drive.mount('/content/drive')
        zip_file = os.path.join('/content/drive/MyDrive', drive_path, f"{dataset_name}.zip")
        out_dir = os.path.join(local_extract_path, dataset_name)
        os.makedirs(local_extract_path, exist_ok=True)
        if not os.path.exists(out_dir):
            with zipfile.ZipFile(zip_file, 'r') as z:
                z.extractall(local_extract_path)
        return out_dir
    finally:
        drive.flush_and_unmount()

# -----------------------------
# CombinedDataset
# -----------------------------
class CombinedDataset(Dataset):
    def __init__(self, image_dir, transform, is_fake, fake_method_label=None, data_fraction=1.0, composite_label=None):
        self.image_dir = image_dir
        self.image_names = [f for f in os.listdir(image_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))]
        if data_fraction < 1.0:
            self.image_names = self.image_names[:int(len(self.image_names)*data_fraction)]
        self.transform = transform
        self.composite_label = composite_label if is_fake else -1
        self.detection_label = 1 if is_fake else 0
        self.fake_method_label = fake_method_label if is_fake else -1

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        try:
            img_path = os.path.join(self.image_dir, self.image_names[idx])
            img = Image.open(img_path).convert('RGB')
            return self.transform(img), self.composite_label, self.detection_label, self.fake_method_label
        except Exception as e:
            print(f"[Warning] Skipping unreadable image: {self.image_names[idx]} – {e}")
            # Recursively try the next image
            return self.__getitem__((idx + 1) % len(self))

# -----------------------------
# FeatureExtractor (DINOv2)
# -----------------------------
class FeatureExtractor(nn.Module):
    def __init__(self, version="l"):
        super().__init__()
        self.net = torch.hub.load('facebookresearch/dinov2', f'dinov2_vit{version}14')
        self.net.eval()
    def forward(self, x):
        with torch.no_grad():
            return self.net(x)

# -----------------------------
# Extract + cache features
# -----------------------------
def extract_features_and_labels_composite(loader, feat_ext, device):
    feats, comps, dets, fms = [], [], [], []
    for imgs, c, d, f in tqdm(loader, desc="Extracting Features"):
        imgs = imgs.to(device)
        out = feat_ext(imgs).cpu()
        feats.append(out)
        comps.append(torch.tensor(c))
        dets.append(torch.tensor(d))
        fms.append(torch.tensor(f))
    return (torch.cat(feats), torch.cat(comps),
            torch.cat(dets), torch.cat(fms))

def get_features_for_dataset(ds, feat_ext, device, cache_path, batch_size=32):
    if os.path.exists(cache_path):
        data = np.load(cache_path, allow_pickle=True)
        return (torch.tensor(data['features']),
                torch.tensor(data['comp_labels']),
                torch.tensor(data['det_labels']),
                torch.tensor(data['fake_method_labels']))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    feats, comps, dets, fms = extract_features_and_labels_composite(loader, feat_ext, device)
    np.savez(cache_path,
             features=feats.numpy(),
             comp_labels=comps.numpy(),
             det_labels=dets.numpy(),
             fake_method_labels=fms.numpy())
    return feats, comps, dets, fms

# -----------------------------
# Wrap precomputed embeddings
# -----------------------------
class FeatureDataset(Dataset):
    def __init__(self, features, comp_labels, det_labels, fm_labels):
        self.features = features
        self.comp_labels = comp_labels
        self.det_labels = det_labels
        self.fm_labels = fm_labels
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.comp_labels[idx], self.det_labels[idx], self.fm_labels[idx]

# -----------------------------
# Linear projection + Loss
# -----------------------------
class LinearModel(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        # Six-layer MLP, all hidden dims = out_dim
        self.fc1 = nn.Linear(in_dim,  out_dim)
        self.fc2 = nn.Linear(out_dim, out_dim)
        self.fc3 = nn.Linear(out_dim, out_dim)
        self.fc4 = nn.Linear(out_dim, out_dim)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.act(self.fc3(x))
        return self.fc4(x)


class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        # features: [B, D], labels: [B]
        device = features.device
        labels = labels.contiguous().view(-1, 1)            # [B,1]
        feats = F.normalize(features, dim=1)               # [B,D]

        # compute similarity matrix
        logits = torch.matmul(feats, feats.T) / self.temperature  # [B,B]

        # mask out self-comparisons
        logits_mask = torch.ones_like(logits, device=device)
        logits_mask.fill_diagonal_(0)

        # log‐probabilities
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-12)

        # only keep positives in numerator
        pos_mask = (labels == labels.T).float().to(device) * logits_mask

        # per-sample loss
        mean_log_prob_pos = (pos_mask * log_prob).sum(1) / (pos_mask.sum(1) + 1e-12)

        # *here* we take the mean over the batch to produce a single scalar
        loss = - mean_log_prob_pos.mean()
        return loss


# -----------------------------
# Train with KMeans pseudo-labels
# -----------------------------
def train_linear_layer(loader, model, opt, loss_fn, device,
                       n_real_clusters, epochs, kmeans_bs):
    model.train()
    for ep in range(epochs):
        tot_loss = 0
        kmeans = MiniBatchKMeans(n_clusters=n_real_clusters, batch_size=kmeans_bs)
        for feats, comps, _, _ in tqdm(loader, desc=f"Epoch {ep+1}/{epochs}"):
            feats = feats.to(device)
            comps = comps.to(device)
            proj = model(feats)
            lbls = torch.empty_like(comps)
            real_mask = (comps == -1)
            if real_mask.sum()>0:
                real_np = feats[real_mask].cpu().numpy()
                if len(real_np) >= n_real_clusters:
                    kmeans.partial_fit(real_np)
                    pseudo = kmeans.predict(real_np)
                else:
                    pseudo = np.zeros(len(real_np),dtype=int)
                pseudo = torch.tensor(pseudo, device=device, dtype=comps.dtype)
                lbls[real_mask] = pseudo
            lbls[~real_mask] = comps[~real_mask]
            loss = loss_fn(proj, lbls)
            opt.zero_grad(); loss.backward(); opt.step()
            tot_loss += loss.item()
        if ep % 10 == 0:
          print(f"Epoch {ep+1} avg loss: {tot_loss/len(loader):.4f}")

# -----------------------------
# Outlier stats & eval
# -----------------------------
def compute_cluster_stats(loader, model, device):
    model.eval()
    clust = {}
    with torch.no_grad():
        for feats, comps, _, _ in loader:
            feats = feats.to(device)
            pr = model(feats).cpu().numpy()
            for v,c in zip(pr, comps.numpy()):
                clust.setdefault(int(c), []).append(v)
    stats = {}
    for c,vecs in clust.items():
        arr = np.stack(vecs)
        stats[c] = {'mean':arr.mean(0), 'std':arr.std(0)}
    return stats

def evaluate_with_outlier(loader, model, stats, device):
    model.eval()
    cd = 0      # correct detections
    tot = 0
    cm = 0      # correct methods
    tm = 0

    # Track per-class detection accuracy
    real_tot = 0
    real_correct = 0
    fake_tot = 0
    fake_correct = 0

    with torch.no_grad():
        for feats, _, dets, fms in loader:
            feats = feats.to(device)
            pr = model(feats).cpu().numpy()

            for v, td, tf in zip(pr, dets.numpy(), fms.numpy()):
                tot += 1

                # OUTLIER DETECTION
                inside = any(
                    np.all(np.abs(v - s['mean']) <= 3 * s['std'])
                    for s in stats.values()
                )
                if not inside:
                    # Outlier → fake, unknown method
                    pd, pm = 1, -1
                else:
                    # Nearest‐centroid fallback
                    dists = {c: np.linalg.norm(v - s['mean']) for c, s in stats.items()}
                    best = min(dists, key=dists.get)
                    if best == -1:
                        pd, pm = 0, -1    # real
                    else:
                        pd, pm = 1, best  # fake, method=best

                # Detection accuracy
                if pd == td:
                    cd += 1

                # Real vs fake breakdown
                if td == 0:
                    real_tot += 1
                    if pd == 0:
                        real_correct += 1
                else:
                    fake_tot += 1
                    if pd == 1:
                        fake_correct += 1

                # Attribution accuracy (for fakes)
                if td == 1:
                    tm += 1
                    if pm == tf:
                        cm += 1

    overall_acc = cd / tot
    method_acc  = cm / tm      if tm      > 0 else 0.0
    real_acc    = real_correct / real_tot if real_tot > 0 else 0.0
    fake_acc    = fake_correct / fake_tot if fake_tot > 0 else 0.0

    return overall_acc, method_acc, real_acc, fake_acc



# -----------------------------
# t-SNE plot
# -----------------------------
def plot_tsne_multiclass(datasets, label_map, model, device, name, out_dir):
    all_f, all_l = [], []
    for ds, attr in datasets:
        for b in DataLoader(ds, 256, False):
            x, c, d, fm = b
            with torch.no_grad():
                pr = model(x.to(device)).cpu().numpy()
            all_f.append(pr)
            all_l.append({'composite':c.numpy(), 'detection':d.numpy(), 'fake_method':fm.numpy()}[attr])
    X = np.vstack(all_f); y = np.hstack(all_l)
    ts = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X)
    plt.figure(figsize=(8,8))
    for lbl, nm in label_map.items():
        m = (y==lbl)
        plt.scatter(ts[m,0], ts[m,1], label=nm, alpha=0.6, s=8)
    plt.legend(bbox_to_anchor=(1.02,1), loc='upper left')
    plt.title(f"t-SNE – held-out → {name}", fontsize=18)
    plt.xlabel("dim1", fontsize=14)
    plt.ylabel("dim2", fontsize=14)
    plt.tight_layout()
    os.makedirs(out_dir, exist_ok=True)
    plt.savefig(os.path.join(out_dir, f"tsne_{name}.png"), dpi=300)
    plt.show()
    plt.close()

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import zipfile
from google.colab import drive
from sklearn.cluster import MiniBatchKMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# -----------------------------
# Helper: Mount & extract dataset from Drive
# -----------------------------
def prepare_dataset_from_drive(drive_path, local_extract_path, dataset_name):
    try:
        drive.mount('/content/drive')
        zip_file = os.path.join('/content/drive/MyDrive', drive_path, f"{dataset_name}.zip")
        out_dir = os.path.join(local_extract_path, dataset_name)
        os.makedirs(local_extract_path, exist_ok=True)
        if not os.path.exists(out_dir):
            with zipfile.ZipFile(zip_file, 'r') as z:
                z.extractall(local_extract_path)
        return out_dir
    finally:
        drive.flush_and_unmount()

# -----------------------------
# CombinedDataset
# -----------------------------
class CombinedDataset(Dataset):
    def __init__(self, image_dir, transform, is_fake, fake_method_label=None, data_fraction=1.0, composite_label=None):
        self.image_dir = image_dir
        self.image_names = [f for f in os.listdir(image_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))]
        if data_fraction < 1.0:
            self.image_names = self.image_names[:int(len(self.image_names)*data_fraction)]
        self.transform = transform
        self.composite_label = composite_label if is_fake else -1
        self.detection_label = 1 if is_fake else 0
        self.fake_method_label = fake_method_label if is_fake else -1

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        try:
            img_path = os.path.join(self.image_dir, self.image_names[idx])
            img = Image.open(img_path).convert('RGB')
            return self.transform(img), self.composite_label, self.detection_label, self.fake_method_label
        except Exception as e:
            print(f"[Warning] Skipping unreadable image: {self.image_names[idx]} – {e}")
            # Recursively try the next image
            return self.__getitem__((idx + 1) % len(self))

# -----------------------------
# FeatureExtractor (DINOv2)
# -----------------------------
class FeatureExtractor(nn.Module):
    def __init__(self, model_type="dinov2_vitl14"):
        super().__init__()
        self.model_type = model_type.lower()
        self.is_clip = "vit-" in self.model_type.lower()  # crude CLIP check

        if self.is_clip:
            import open_clip

            # Clean name and tag
            model_name = model_type
            clip_tag_map = {
                "ViT-B-32": "laion400m_e31",
                "ViT-L-14": "openai"
            }



            tag = clip_tag_map.get(model_type, "openai")


            model, _, preprocess = open_clip.create_model_and_transforms(
                model_name=model_name, pretrained=tag
            )
            self.model = model.eval()
            self.preprocess = preprocess

        else:
            self.model = torch.hub.load('facebookresearch/dinov2', model_type).eval()
            self.preprocess = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])
            ])

    def forward(self, x):
        with torch.no_grad():
            if self.is_clip:
                x = self.model.visual(x)
            else:
                out = self.model(x)
                if isinstance(out, torch.Tensor):
                    return out
                elif isinstance(out, dict) and 'x_norm_clstoken' in out:
                    return out['x_norm_clstoken']
                raise ValueError("Unexpected DINOv2 output shape")
            return x



# -----------------------------
# Extract + cache features
# -----------------------------
def extract_features_and_labels_composite(loader, feat_ext, device):
    feats, comps, dets, fms = [], [], [], []
    for imgs, c, d, f in tqdm(loader, desc="Extracting Features"):
        imgs = imgs.to(device)
        out = feat_ext(imgs).cpu()
        feats.append(out)
        comps.append(torch.tensor(c))
        dets.append(torch.tensor(d))
        fms.append(torch.tensor(f))
    return (torch.cat(feats), torch.cat(comps),
            torch.cat(dets), torch.cat(fms))

def get_features_for_dataset(ds, feat_ext, device, cache_path, batch_size=32):
    if os.path.exists(cache_path):
        data = np.load(cache_path, allow_pickle=True)
        return (torch.tensor(data['features']),
                torch.tensor(data['comp_labels']),
                torch.tensor(data['det_labels']),
                torch.tensor(data['fake_method_labels']))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    feats, comps, dets, fms = extract_features_and_labels_composite(loader, feat_ext, device)
    np.savez(cache_path,
             features=feats.numpy(),
             comp_labels=comps.numpy(),
             det_labels=dets.numpy(),
             fake_method_labels=fms.numpy())
    return feats, comps, dets, fms

# -----------------------------
# Wrap precomputed embeddings
# -----------------------------
class FeatureDataset(Dataset):
    def __init__(self, features, comp_labels, det_labels, fm_labels):
        self.features = features
        self.comp_labels = comp_labels
        self.det_labels = det_labels
        self.fm_labels = fm_labels
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.comp_labels[idx], self.det_labels[idx], self.fm_labels[idx]

# -----------------------------
# Linear projection + Loss
# -----------------------------
class LinearModel(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        # Six-layer MLP, all hidden dims = out_dim
        self.fc1 = nn.Linear(in_dim,  out_dim)
        self.fc2 = nn.Linear(out_dim, out_dim)
        self.fc3 = nn.Linear(out_dim, out_dim)
        self.fc4 = nn.Linear(out_dim, out_dim)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.act(self.fc3(x))
        return self.fc4(x)


class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        # features: [B, D], labels: [B]
        device = features.device
        labels = labels.contiguous().view(-1, 1)            # [B,1]
        feats = F.normalize(features, dim=1)               # [B,D]

        # compute similarity matrix
        logits = torch.matmul(feats, feats.T) / self.temperature  # [B,B]

        # mask out self-comparisons
        logits_mask = torch.ones_like(logits, device=device)
        logits_mask.fill_diagonal_(0)

        # log‐probabilities
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-12)

        # only keep positives in numerator
        pos_mask = (labels == labels.T).float().to(device) * logits_mask

        # per-sample loss
        mean_log_prob_pos = (pos_mask * log_prob).sum(1) / (pos_mask.sum(1) + 1e-12)

        # *here* we take the mean over the batch to produce a single scalar
        loss = - mean_log_prob_pos.mean()
        return loss


# -----------------------------
# Train with KMeans pseudo-labels
# -----------------------------
def train_linear_layer(loader, model, opt, loss_fn, device,
                       n_real_clusters, epochs, kmeans_bs):
    model.train()
    for ep in range(epochs):
        tot_loss = 0
        kmeans = MiniBatchKMeans(n_clusters=n_real_clusters, batch_size=kmeans_bs)
        for feats, comps, _, _ in tqdm(loader, desc=f"Epoch {ep+1}/{epochs}"):
            feats = feats.to(device)
            comps = comps.to(device)
            proj = model(feats)
            lbls = torch.empty_like(comps)
            real_mask = (comps == -1)
            if real_mask.sum()>0:
                real_np = feats[real_mask].cpu().numpy()
                if len(real_np) >= n_real_clusters:
                    kmeans.partial_fit(real_np)
                    pseudo = kmeans.predict(real_np)
                else:
                    pseudo = np.zeros(len(real_np),dtype=int)
                pseudo = torch.tensor(pseudo, device=device, dtype=comps.dtype)
                lbls[real_mask] = pseudo
            lbls[~real_mask] = comps[~real_mask]
            loss = loss_fn(proj, lbls)
            opt.zero_grad(); loss.backward(); opt.step()
            tot_loss += loss.item()
        if ep % 10 == 0:
          print(f"Epoch {ep+1} avg loss: {tot_loss/len(loader):.4f}")

# -----------------------------
# Outlier stats & eval
# -----------------------------
def compute_cluster_stats(loader, model, device):
    model.eval()
    clust = {}
    with torch.no_grad():
        for feats, comps, _, _ in loader:
            feats = feats.to(device)
            pr = model(feats).cpu().numpy()
            for v,c in zip(pr, comps.numpy()):
                clust.setdefault(int(c), []).append(v)
    stats = {}
    for c,vecs in clust.items():
        arr = np.stack(vecs)
        stats[c] = {'mean':arr.mean(0), 'std':arr.std(0)}
    return stats

def evaluate_with_outlier(loader, model, stats, device):
    model.eval()
    cd = 0      # correct detections
    tot = 0
    cm = 0      # correct methods
    tm = 0

    # Track per-class detection accuracy
    real_tot = 0
    real_correct = 0
    fake_tot = 0
    fake_correct = 0

    with torch.no_grad():
        for feats, _, dets, fms in loader:
            feats = feats.to(device)
            pr = model(feats).cpu().numpy()

            for v, td, tf in zip(pr, dets.numpy(), fms.numpy()):
                tot += 1

                # OUTLIER DETECTION
                inside = any(
                    np.all(np.abs(v - s['mean']) <= 3 * s['std'])
                    for s in stats.values()
                )
                if not inside:
                    # Outlier → fake, unknown method
                    pd, pm = 1, -1
                else:
                    # Nearest‐centroid fallback
                    dists = {c: np.linalg.norm(v - s['mean']) for c, s in stats.items()}
                    best = min(dists, key=dists.get)
                    if best == -1:
                        pd, pm = 0, -1    # real
                    else:
                        pd, pm = 1, best  # fake, method=best

                # Detection accuracy
                if pd == td:
                    cd += 1

                # Real vs fake breakdown
                if td == 0:
                    real_tot += 1
                    if pd == 0:
                        real_correct += 1
                else:
                    fake_tot += 1
                    if pd == 1:
                        fake_correct += 1

                # Attribution accuracy (for fakes)
                if td == 1:
                    tm += 1
                    if pm == tf:
                        cm += 1

    overall_acc = cd / tot
    method_acc  = cm / tm      if tm      > 0 else 0.0
    real_acc    = real_correct / real_tot if real_tot > 0 else 0.0
    fake_acc    = fake_correct / fake_tot if fake_tot > 0 else 0.0

    return overall_acc, method_acc, real_acc, fake_acc



# -----------------------------
# t-SNE plot
# -----------------------------
def plot_tsne_multiclass(datasets, label_map, model, device, name, out_dir):
    all_f, all_l = [], []
    for ds, attr in datasets:
        for b in DataLoader(ds, 256, False):
            x, c, d, fm = b
            with torch.no_grad():
                pr = model(x.to(device)).cpu().numpy()
            all_f.append(pr)
            all_l.append({'composite':c.numpy(), 'detection':d.numpy(), 'fake_method':fm.numpy()}[attr])
    X = np.vstack(all_f); y = np.hstack(all_l)
    ts = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X)
    plt.figure(figsize=(8,8))
    for lbl, nm in label_map.items():
        m = (y==lbl)
        plt.scatter(ts[m,0], ts[m,1], label=nm, alpha=0.6, s=8)
    plt.legend(bbox_to_anchor=(1.02,1), loc='upper left')
    plt.title(f"t-SNE – held-out → {name}", fontsize=18)
    plt.xlabel("dim1", fontsize=14)
    plt.ylabel("dim2", fontsize=14)
    plt.tight_layout()
    os.makedirs(out_dir, exist_ok=True)
    plt.savefig(os.path.join(out_dir, f"tsne_{name}.png"), dpi=300)
    plt.show()
    plt.close()

# -----------------------------
# Main
# -----------------------------
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbones = [
        #"dinov2_vitl14",      # DINOv2 large
        "dinov2_vitb14",      # DINOv2 base
        "dinov2_vits14",      # DINOv2 tiny
        #"ViT-B-32",
        #"ViT-L-14"
    ]


    results = {}

    for bb in backbones:
        print(f"\n=== Running with backbone: {bb} ===\n")

        # Dirs and model
        drive_path = "datasets"
        local_ex   = "/content/dataset_extracted"
        cache_dir  = os.path.join(local_ex, "features_cache", bb.replace("/", "_"))
        os.makedirs(cache_dir, exist_ok=True)

        # Feature extractor
        feat_ext = FeatureExtractor(bb).to(device)

        # Datasets (as in your original code)
        real_name  = "real"
        fake_names = ["biggan_fake", "cyclegan_fake", "ADM_fake", "progan_fake", "DALLE2_fake", "Glide_fake",
                      "gaugan_fake", "stargan_fake", "stylegan_fake", "stable_diffusion_v_1_4_fake",
                      "stylegan2_fake", "sd_xl_fake", "VQDM_fake", "whichfaceisreal_fake",
                      "stable_diffusion_v_1_5_fake", "Midjourney_fake"]
        df=1.0; ts=0.2; bs=32
        temp=0.07; cbs=256; eps=10; lr=1e-3
        total_clusters=100; nr_fake=len(fake_names)
        nreal = total_clusters - nr_fake
        num_real_images = 1000
        num_per_gen     = 1000

        tfm = feat_ext.preprocess

        rp = prepare_dataset_from_drive(drive_path, local_ex, real_name)
        real_imgs = [f for f in os.listdir(rp) if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:num_real_images]
        rd = CombinedDataset(rp, tfm, False)
        rd.image_names = real_imgs

        r_cache = os.path.join(cache_dir, f"{real_name}.npz")
        rF, rC, rD, rFM = get_features_for_dataset(rd, feat_ext, device, r_cache, bs)

        allf = []
        for i,nm in enumerate(fake_names):
            p = prepare_dataset_from_drive(drive_path, local_ex, nm)
            fake_imgs = [f for f in os.listdir(p) if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:num_per_gen]
            ds = CombinedDataset(p, tfm, True, fake_method_label=i, composite_label=nreal+i)
            ds.image_names = fake_imgs
            cpath = os.path.join(cache_dir, f"{nm}.npz")
            F, C, D, FM = get_features_for_dataset(ds, feat_ext, device, cpath, bs)
            allf.append({'name':nm,'F':F,'C':C,'D':D,'FM':FM})

        # Leave-one-out results
        results[bb] = []
        for idx in range(len(fake_names)):
            held = allf[idx]
            seen = [f for j,f in enumerate(allf) if j!=idx]

            TF  = torch.cat([rF] + [s['F'] for s in seen], 0)
            TC  = torch.cat([rC] + [s['C'] for s in seen], 0)
            TD  = torch.cat([rD] + [s['D'] for s in seen], 0)
            TFM = torch.cat([rFM]+ [s['FM'] for s in seen],0)

            tr,te = train_test_split(range(TF.size(0)), test_size=ts, random_state=42)
            train_ds = FeatureDataset(TF[tr], TC[tr], TD[tr], TFM[tr])
            seen_ds  = FeatureDataset(TF[te], TC[te], TD[te], TFM[te])

            uid = list(range(held['F'].size(0)))
            _, ute = train_test_split(uid, test_size=ts, random_state=42)
            unseen_fake_ds = FeatureDataset(held['F'][ute], held['C'][ute], held['D'][ute], held['FM'][ute])

            real_ids = list(range(rF.size(0)))
            _, real_ute = train_test_split(real_ids, test_size=ts, random_state=42)
            unseen_real_ds = FeatureDataset(rF[real_ute], rC[real_ute], rD[real_ute], rFM[real_ute])
            unseen_ds = torch.utils.data.ConcatDataset([unseen_fake_ds, unseen_real_ds])

            trL = DataLoader(train_ds, batch_size=cbs, shuffle=True)
            sdL = DataLoader(seen_ds,  batch_size=cbs, shuffle=False)
            udL = DataLoader(unseen_ds,batch_size=cbs, shuffle=True)

            lm = LinearModel(TF.size(1), 512).to(device)
            opt = torch.optim.Adam(lm.parameters(), lr=lr)
            loss_fn = SupervisedContrastiveLoss(temp)
            train_linear_layer(trL, lm, opt, loss_fn, device, nreal, eps, cbs)

            stats = compute_cluster_stats(trL, lm, device)
            a_s, m_s, r_s, f_s = evaluate_with_outlier(sdL, lm, stats, device)
            a_u, m_u, r_u, f_u = evaluate_with_outlier(udL, lm, stats, device)

            results[bb].append({
                'held_out': held['name'],
                'seen_acc': a_s,
                'unseen_acc': a_u,
                'seen_real': r_s,
                'seen_fake': f_s,
                'unseen_real': r_u,
                'unseen_fake': f_u,
                'method_acc_seen': m_s,
                'method_acc_unseen': m_u
            })

            print(f"\n{bb} → Held-out {held['name']}:")
            print(f"  Seen   detect {a_s*100:.2f}% method {m_s*100:.2f}%")
            print(f"  Unseen detect {a_u*100:.2f}% method {m_u*100:.2f}%")

    # Save final result summary
    import json
    with open(os.path.join(local_ex, "backbone_results.json"), "w") as f:
        json.dump(results, f, indent=2)


if __name__ == "__main__":
    main()

In [ ]:
!pip install open-clip-torch


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import zipfile
from google.colab import drive
from sklearn.cluster import MiniBatchKMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # params
    drive_path = "datasets"
    local_ex   = "/content/dataset_extracted"
    real_name  = "real"
    fake_names = ["biggan_fake", "cyclegan_fake", "ADM_fake", "progan_fake", "DALLE2_fake", "Glide_fake",
                  "gaugan_fake", "stargan_fake", "stylegan_fake", "stable_diffusion_v_1_4_fake",
                  "stylegan2_fake", "sd_xl_fake", "VQDM_fake", "whichfaceisreal_fake",
                  "stable_diffusion_v_1_5_fake", "Midjourney_fake"]
    df=1.0; ts=0.2; bs=32
    temp=0.07; cbs=256; eps=10; lr=1e-3
    total_clusters=100; nr_fake=len(fake_names)
    nreal = total_clusters - nr_fake
    num_real_images = 5000        # Number of real images to include
    num_per_gen     = 5000         # Number of fake images per generator

    tfm = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    cache_dir = os.path.join(local_ex, "features_cache")
    os.makedirs(cache_dir, exist_ok=True)

    feat_ext = FeatureExtractor().to(device)

    # real
    rp = prepare_dataset_from_drive(drive_path, local_ex, real_name)
    real_imgs = os.listdir(rp)
    real_imgs = [f for f in real_imgs if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    real_imgs = real_imgs[:num_real_images]
    rd = CombinedDataset(rp, tfm, False)
    rd.image_names = real_imgs

    r_cache = os.path.join(cache_dir, f"{real_name}.npz")
    rF, rC, rD, rFM = get_features_for_dataset(rd, feat_ext, device, r_cache, bs)

    # fakes
    allf = []
    for i,nm in enumerate(fake_names):
        p = prepare_dataset_from_drive(drive_path, local_ex, nm)
        fake_imgs = os.listdir(p)
        fake_imgs = [f for f in fake_imgs if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        fake_imgs = fake_imgs[:num_per_gen]
        ds = CombinedDataset(p, tfm, True, fake_method_label=i,
                            composite_label=nreal+i)
        ds.image_names = fake_imgs
        cpath = os.path.join(cache_dir, f"{nm}.npz")
        F, C, D, FM = get_features_for_dataset(ds, feat_ext, device, cpath, bs)
        allf.append({'name':nm,'F':F,'C':C,'D':D,'FM':FM})

    import random
    import csv

    # Evaluation parameters
    num_permutations = 15
    N_values = [13]
    results_path = os.path.join(local_ex, "leave_n_out_results.csv")

    # Write header to CSV
    with open(results_path, "w", newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["N", "Seen_Detect_Acc", "Unseen_Detect_Acc", "Seen_Method_Acc", "Unseen_Method_Acc"])

    for N in N_values:
        seen_accs = []
        unseen_accs = []
        seen_method_accs = []
        unseen_method_accs = []

        for _ in range(num_permutations):
            held_idxs = random.sample(range(len(fake_names)), N)
            seen = [f for j,f in enumerate(allf) if j not in held_idxs]
            held = [f for j,f in enumerate(allf) if j in held_idxs]

            # Assemble seen dataset
            TF  = torch.cat([rF] + [s['F'] for s in seen], 0)
            TC  = torch.cat([rC] + [s['C'] for s in seen], 0)
            TD  = torch.cat([rD] + [s['D'] for s in seen], 0)
            TFM = torch.cat([rFM]+ [s['FM'] for s in seen],0)

            ids = list(range(TF.size(0)))
            tr,te = train_test_split(ids, test_size=ts, random_state=42)
            train_ds = FeatureDataset(TF[tr], TC[tr], TD[tr], TFM[tr])
            seen_ds  = FeatureDataset(TF[te], TC[te], TD[te], TFM[te])

            # Unseen set: mix of all held-out fakes + sampled reals
            heldF = torch.cat([h['F'] for h in held], 0)
            heldC = torch.cat([h['C'] for h in held], 0)
            heldD = torch.cat([h['D'] for h in held], 0)
            heldFM = torch.cat([h['FM'] for h in held], 0)

            uid = list(range(heldF.size(0)))
            _, ute = train_test_split(uid, test_size=ts, random_state=42)
            unseen_fake_ds = FeatureDataset(heldF[ute], heldC[ute], heldD[ute], heldFM[ute])

            real_ids = list(range(rF.size(0)))
            _, real_ute = train_test_split(real_ids, test_size=ts, random_state=42)
            unseen_real_ds = FeatureDataset(rF[real_ute], rC[real_ute], rD[real_ute], rFM[real_ute])

            unseen_ds = torch.utils.data.ConcatDataset([unseen_fake_ds, unseen_real_ds])

            # Train and evaluate
            trL = DataLoader(train_ds, batch_size=cbs, shuffle=True)
            sdL = DataLoader(seen_ds,  batch_size=cbs, shuffle=False)
            udL = DataLoader(unseen_ds,batch_size=cbs, shuffle=False)

            lm = LinearModel(TF.size(1), 512).to(device)
            opt = torch.optim.Adam(lm.parameters(), lr=lr)
            loss_fn = SupervisedContrastiveLoss(temp)
            train_linear_layer(trL, lm, opt, loss_fn, device, nreal, eps, cbs)

            stats = compute_cluster_stats(trL, lm, device)
            a_s, m_s, _, _ = evaluate_with_outlier(sdL, lm, stats, device)
            a_u, m_u, _, _ = evaluate_with_outlier(udL, lm, stats, device)

            seen_accs.append(a_s)
            unseen_accs.append(a_u)
            seen_method_accs.append(m_s)
            unseen_method_accs.append(m_u)

        # Average and save results
        mean_seen_acc = 100 * np.mean(seen_accs)
        mean_unseen_acc = 100 * np.mean(unseen_accs)
        mean_seen_meth = 100 * np.mean(seen_method_accs)
        mean_unseen_meth = 100 * np.mean(unseen_method_accs)

        print(f"\nLeave-{N}-Out ({num_permutations} runs):")
        print(f"  Seen detect acc: {mean_seen_acc:.2f}%")
        print(f"  Unseen detect acc: {mean_unseen_acc:.2f}%")

        with open(results_path, "a", newline='') as f:
            writer = csv.writer(f)
            writer.writerow([N, mean_seen_acc, mean_unseen_acc, mean_seen_meth, mean_unseen_meth])


if __name__ == "__main__":
    main()

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset, ConcatDataset, Subset, TensorDataset
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import zipfile
import csv
from google.colab import drive

# --- Helper

def prepare_dataset_from_drive(drive_path, local_extract_path, dataset_name):
    try:
        drive.mount('/content/drive')
        zip_file = os.path.join('/content/drive/MyDrive', drive_path, f"{dataset_name}.zip")
        out_dir = os.path.join(local_extract_path, dataset_name)
        os.makedirs(local_extract_path, exist_ok=True)
        if not os.path.exists(out_dir):
            with zipfile.ZipFile(zip_file, 'r') as z:
                z.extractall(local_extract_path)
        return out_dir
    finally:
        drive.flush_and_unmount()

# --- Dataset
class CombinedDataset(Dataset):
    def __init__(self, image_dir, transform, label, max_images=None):
        self.image_dir = image_dir
        self.image_names = [f for f in os.listdir(image_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))]
        random.shuffle(self.image_names)
        if max_images is not None:
            self.image_names = self.image_names[:max_images]
        self.transform = transform
        self.label = label

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        try:
            img_path = os.path.join(self.image_dir, self.image_names[idx])
            img = Image.open(img_path).convert('RGB')
            return self.transform(img), self.label
        except:
            return self.__getitem__((idx + 1) % len(self))

# --- Feature extractor
class FeatureExtractor(nn.Module):
    def __init__(self, version="l"):
        super().__init__()
        self.net = torch.hub.load('facebookresearch/dinov2', f'dinov2_vit{version}14')
        self.net.eval()

    def forward(self, x):
        with torch.no_grad():
            out = self.net(x)
            if isinstance(out, torch.Tensor):
                return out
            elif isinstance(out, dict) and 'x_norm_clst' in out:
                return out['x_norm_clst']
            raise ValueError("Unexpected DINOv2 output")

# --- Linear classifier
class LinearClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

# --- Feature caching

def extract_or_load_features(dataset_name, dataset, feat_ext, device, cache_dir='features_cache'):
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = os.path.join(cache_dir, f"{dataset_name}.npz")
    if os.path.exists(cache_path):
        npz = np.load(cache_path)
        return torch.tensor(npz['feats']), torch.tensor(npz['labels'])
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    feats, labs = [], []
    for x, y in tqdm(loader, desc=f"FeatExtract: {dataset_name}", leave=False):
        x = x.to(device)
        f = feat_ext(x)
        feats.append(f.cpu()); labs.append(y)
    feats = torch.cat(feats)
    labs = torch.cat(labs)
    np.savez(cache_path, feats=feats.numpy(), labels=labs.numpy())
    return feats, labs

# --- Main

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    drive_path = "datasets"
    local_ex = "/content/dataset_extracted"
    real_name = "real"
    fake_names = [
        "biggan_fake", "cyclegan_fake", "ADM_fake", "progan_fake", "DALLE2_fake", "Glide_fake",
        "gaugan_fake", "stargan_fake", "stylegan_fake", "stable_diffusion_v_1_4_fake",
        "stylegan2_fake", "sd_xl_fake", "VQDM_fake", "whichfaceisreal_fake",
        "stable_diffusion_v_1_5_fake", "Midjourney_fake"
    ]
    batch_size = 32
    num_epochs = 10
    lr = 1e-3
    max_real = 100
    max_fake_per_gen = 100
    n_classes = 2

    tfm = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])

    feat_ext = FeatureExtractor().to(device)

    real_path = prepare_dataset_from_drive(drive_path, local_ex, real_name)
    real_ds = CombinedDataset(real_path, tfm, label=0, max_images=max_real)
    real_train_idx, real_test_idx = train_test_split(list(range(len(real_ds))), test_size=0.2, random_state=42)
    real_train = Subset(real_ds, real_train_idx)
    real_test = Subset(real_ds, real_test_idx)

    fake_paths = {name: prepare_dataset_from_drive(drive_path, local_ex, name) for name in fake_names}

    for idx, heldout_name in enumerate(fake_names):
        print(f"\n[LOO] Holding out: {heldout_name}")
        train_fakes = [name for name in fake_names if name != heldout_name]

        train_fake_ds = ConcatDataset([
            CombinedDataset(fake_paths[name], tfm, label=1, max_images=max_fake_per_gen)
            for name in train_fakes
        ])

        train_comb = ConcatDataset([real_train, train_fake_ds])
        x_train, y_train = extract_or_load_features(f"train_comb_{heldout_name}", train_comb, feat_ext, device)

        clf = LinearClassifier(x_train.shape[1], num_classes=n_classes).to(device)
        opt = torch.optim.Adam(clf.parameters(), lr=lr)
        crit = nn.CrossEntropyLoss()

        clf.train()
        train_tensor_ds = TensorDataset(x_train, y_train)
        train_feat_loader = DataLoader(train_tensor_ds, batch_size=batch_size, shuffle=True)
        for epoch in range(num_epochs):
            total_loss = 0.0
            for x, y in train_feat_loader:
                x, y = x.to(device), y.to(device)
                opt.zero_grad()
                loss = crit(clf(x), y)
                loss.backward()
                opt.step()
                total_loss += loss.item() * x.size(0)
            print(f"  Epoch {epoch+1}/{num_epochs} → train loss {total_loss / len(train_tensor_ds):.4f}")

        seen_fake_ds = ConcatDataset([
            CombinedDataset(fake_paths[name], tfm, label=1, max_images=max_fake_per_gen)
            for name in train_fakes
        ])
        seen_comb = ConcatDataset([real_test, seen_fake_ds])
        x_seen, y_seen = extract_or_load_features(f"seen_comb_{heldout_name}", seen_comb, feat_ext, device)

        heldout_fake_ds = CombinedDataset(fake_paths[heldout_name], tfm, label=1, max_images=max_fake_per_gen)
        min_len = min(len(real_test), len(heldout_fake_ds))
        heldout_fake_ds = Subset(heldout_fake_ds, list(range(min_len)))
        real_test_bal = Subset(real_test, list(range(min_len)))
        unseen_comb = ConcatDataset([real_test_bal, heldout_fake_ds])
        x_unseen, y_unseen = extract_or_load_features(f"unseen_comb_{heldout_name}", unseen_comb, feat_ext, device)

        def evaluate(x, y):
            clf.eval()
            with torch.no_grad():
                logits = clf(x.to(device))
                preds = logits.argmax(dim=1).cpu()
                acc = (preds == y).float().mean().item()
                return acc

        acc_seen = evaluate(x_seen, y_seen)
        acc_unseen = evaluate(x_unseen, y_unseen)

if __name__ == "__main__":
    main()
